# Notebook 09: Deployment Comparison & Optimization for ESP32-S3

**Purpose:** Comprehensive comparison of Classical ML vs Neural Network deployment strategies

**Objectives:**
1. **Classical ML Path:** Calculate memory requirements for feature extraction + inference
2. **Neural Network Path:** Evaluate quantization strategies (INT8, INT16x8)
3. **Memory Profiling:** Real SRAM/PSRAM usage estimation
4. **Quantization Optimization:** Automatically select best quantization method
5. **Deployment Recommendation:** Data-driven decision for ESP32-S3

**Target Device: XIAO ESP32-S3 Sense**
```
SRAM:   512 KB  (fast, for inference)
PSRAM:  8 MB    (slow, for buffers/weights)
Flash:  8 MB    (model storage)
```

**Deployment Options:**
- **Option A:** Classical ML (emlearn) - XGBoost/RandomForest + feature extraction
- **Option B:** Neural Network (TFLite Micro) - CNN/LSTM with quantization

---

## Section 1: Setup & Configuration

In [1]:
import os, sys
from pathlib import Path
import json
import time
from datetime import datetime
import warnings
import gc
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib

# TensorFlow/Keras
import tensorflow as tf
from tensorflow import keras

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, roc_auc_score
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

print(f'TensorFlow version: {tf.__version__}')
print(f'Keras version: {keras.__version__}')
print('✓ All libraries imported')

TensorFlow version: 2.17.1
Keras version: 3.10.0
✓ All libraries imported


In [2]:
# Project paths
PROJECT_ROOT = Path.cwd().parent
FEATURES_DIR = PROJECT_ROOT / 'data' / 'features'
MODELS_DIR_CLASSICAL = PROJECT_ROOT / 'models' / 'classical'
MODELS_DIR_NEURAL = PROJECT_ROOT / 'models' / 'neural'
TFLITE_DIR = PROJECT_ROOT / 'models' / 'tflite'
RESULTS_DIR = PROJECT_ROOT / 'results'
FIGURES_DIR = RESULTS_DIR / 'figures'

# Create directories
TFLITE_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root:  {PROJECT_ROOT}')
print(f'Models classical: {MODELS_DIR_CLASSICAL}')
print(f'Models neural:    {MODELS_DIR_NEURAL}')
print(f'TFLite dir:       {TFLITE_DIR}')
print(f'Results dir:      {RESULTS_DIR}')

Project root:  /Users/harryirving/Development/projects/ai-ml/BikeAIv5
Models classical: /Users/harryirving/Development/projects/ai-ml/BikeAIv5/models/classical
Models neural:    /Users/harryirving/Development/projects/ai-ml/BikeAIv5/models/neural
TFLite dir:       /Users/harryirving/Development/projects/ai-ml/BikeAIv5/models/tflite
Results dir:      /Users/harryirving/Development/projects/ai-ml/BikeAIv5/results


In [3]:
# ESP32-S3 Memory constraints
ESP32_CONSTRAINTS = {
    'sram_kb': 512,
    'psram_mb': 8,
    'flash_mb': 8,
    'system_overhead_kb': 50,      # WiFi/BLE/RTOS
    'stack_heap_kb': 30,
    'available_sram_kb': 432,       # After system overhead
    'target_model_size_kb': 500,
    'target_inference_ms': 50,
    'target_tensor_arena_kb': 300   # Critical for NN!
}

# Audio configuration (must match training)
AUDIO_CONFIG = {
    'sample_rate': 16000,
    'window_duration_ms': 1000,
    'window_samples': 16000,
    'n_mels': 40,
    'n_fft': 512,
    'hop_length': 160,
    'n_frames': 100  # (16000 - 512) / 160 + 1 ≈ 100
}

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('✓ Configuration loaded')
print(f'\nESP32-S3 Memory Budget:')
print(f'  Available SRAM:   {ESP32_CONSTRAINTS["available_sram_kb"]} KB')
print(f'  Tensor arena max: {ESP32_CONSTRAINTS["target_tensor_arena_kb"]} KB')
print(f'  Model size max:   {ESP32_CONSTRAINTS["target_model_size_kb"]} KB')

✓ Configuration loaded

ESP32-S3 Memory Budget:
  Available SRAM:   432 KB
  Tensor arena max: 300 KB
  Model size max:   500 KB


In [6]:
# Define F1Score metric (needed for loading models saved with custom metrics)
from tensorflow.keras import backend as K

class F1Score(keras.metrics.Metric):
    def __init__(self, name='f1', **kwargs):
        super().__init__(name=name, **kwargs)
        self.precision_metric = keras.metrics.Precision()
        self.recall_metric = keras.metrics.Recall()
    
    def update_state(self, y_true, y_pred, sample_weight=None):
        self.precision_metric.update_state(y_true, y_pred, sample_weight)
        self.recall_metric.update_state(y_true, y_pred, sample_weight)
    
    def result(self):
        p = self.precision_metric.result()
        r = self.recall_metric.result()
        return 2 * ((p * r) / (p + r + K.epsilon()))
    
    def reset_state(self):
        self.precision_metric.reset_state()
        self.recall_metric.reset_state()
    
    def get_config(self):
        config = super().get_config()
        return config

print('✓ Custom F1Score metric defined for model loading')



✓ Custom F1Score metric defined for model loading


## Section 2: Load Best Models from Previous Notebooks

In [9]:
print('\nLOADING BEST MODELS...')
print('='*70)

# Load classical ML results from 06a and 06b
results_06a = RESULTS_DIR / '06a_recommendation_report.json'
results_06b_comparison = RESULTS_DIR / '06b_deployment_comparison_combined.csv'
results_06b_features = RESULTS_DIR / '06b_reduced_feature_sets_combined.json'

if results_06a.exists():
    with open(results_06a, 'r') as f:
        classical_report_06a = json.load(f)
    
    print('✓ Classical ML Report (Notebook 06a):')
    print(f'  Best model: {classical_report_06a["primary_model"]["model_name"]}')
    print(f'  F1 Score:   {classical_report_06a["primary_model"]["test_f1"]:.4f}')
    print(f'  Recall:     {classical_report_06a["primary_model"]["recall"]:.4f}')
    print(f'  Inference:  {classical_report_06a["primary_model"]["inference_ms"]:.3f} ms')
    print(f'  Size:       {classical_report_06a["primary_model"]["size_mb"]:.2f} MB')
    
    # Load 06b optimized feature reduction results
    if results_06b_comparison.exists():
        df_06b = pd.read_csv(results_06b_comparison)
        print('\n✓ Feature Reduction Results (Notebook 06b):')
        print(df_06b.to_string(index=False))
        
        # Use accurate configuration (80 features) - best trade-off
        accurate_row = df_06b[df_06b['Configuration'] == 'ACCURATE'].iloc[0]
        
        classical_report = {
            'best_model': {
                'name': 'XGBoost',
                'configuration': 'accurate_80',
                'test_f1': accurate_row['Test F1'],
                'recall': accurate_row['Recall'],
                'n_features_used': int(accurate_row['Features']),
                'inference_ms': accurate_row['Est. Extraction (ms)'],
                'size_kb': accurate_row['Model Size (KB)']
            }
        }
        
        print(f'\n→ Selected: ACCURATE (80 features) for deployment')
        print(f'   F1: {classical_report["best_model"]["test_f1"]:.4f}')
        print(f'   Recall: {classical_report["best_model"]["recall"]:.4f}')
        print(f'   Features: {classical_report["best_model"]["n_features_used"]}')
        
        # Load feature indices
        if results_06b_features.exists():
            with open(results_06b_features, 'r') as f:
                feature_sets = json.load(f)
            classical_feature_indices = feature_sets['accurate']['indices']
            classical_feature_names = feature_sets['accurate']['names']
            print(f'   ✓ Loaded {len(classical_feature_indices)} feature indices')
        else:
            classical_feature_indices = None
            classical_feature_names = None
    else:
        print('\n⚠️ 06b results not found, using full model from 06a')
        classical_report = {
            'best_model': {
                'name': 'XGBoost',
                'configuration': 'tuned',
                'test_f1': classical_report_06a['primary_model']['test_f1'],
                'recall': classical_report_06a['primary_model']['recall'],
                'n_features_used': 260,  # Full feature set
                'inference_ms': classical_report_06a['primary_model']['inference_ms'],
                'size_kb': classical_report_06a['primary_model']['size_mb'] * 1024
            }
        }
        classical_feature_indices = None
        classical_feature_names = None
    
    # Load actual model
    model_name = f"xgboost_{classical_report['best_model']['configuration']}_combined.pkl"
    classical_model_path = MODELS_DIR_CLASSICAL / model_name
    
    if classical_model_path.exists():
        classical_model = joblib.load(classical_model_path)
        print(f'\n  ✓ Model loaded: {classical_model_path.name}')
        
        # Load scaler
        scaler_path = MODELS_DIR_CLASSICAL / 'scaler_combined.pkl'
        if scaler_path.exists():
            classical_scaler = joblib.load(scaler_path)
            print(f'  ✓ Scaler loaded: {scaler_path.name}')
        else:
            classical_scaler = None
            print(f'  ⚠️ Scaler not found')
    else:
        print(f'\n  ⚠️ Model file not found: {model_name}')
        print(f'  Looking for alternative...')
        # Try tuned version as fallback
        alt_model_path = MODELS_DIR_CLASSICAL / 'xgboost_tuned_combined.pkl'
        if alt_model_path.exists():
            classical_model = joblib.load(alt_model_path)
            print(f'  ✓ Loaded alternative: {alt_model_path.name}')
            scaler_path = MODELS_DIR_CLASSICAL / 'scaler_combined.pkl'
            classical_scaler = joblib.load(scaler_path) if scaler_path.exists() else None
        else:
            classical_model = None
            classical_scaler = None
else:
    print('❌ Run Notebook 06a first')
    classical_report = None
    classical_model = None
    classical_scaler = None
    classical_feature_indices = None
    classical_feature_names = None

print('='*70)


# Load neural network results from 07
nn_report_3ch = RESULTS_DIR / '07_neural_recommendation_report_3ch.json'
nn_report_1ch = RESULTS_DIR / '07_neural_recommendation_report_1ch.json'

# Try 3ch first (usually better accuracy)
if nn_report_3ch.exists():
    with open(nn_report_3ch, 'r') as f:
        neural_report = json.load(f)
    nn_feature_type = '3ch'
    nn_model_path = MODELS_DIR_NEURAL / 'best_neural_model_3ch.keras'
elif nn_report_1ch.exists():
    with open(nn_report_1ch, 'r') as f:
        neural_report = json.load(f)
    nn_feature_type = '1ch'
    nn_model_path = MODELS_DIR_NEURAL / 'best_neural_model_1ch.keras'
else:
    print('❌ Run Notebook 07 first')
    neural_report = None
    nn_model_path = None
    nn_feature_type = None

if neural_report:
    print(f'\n✓ Neural Network Report (Notebook 07 - {nn_feature_type}):')
    print(f'  Best model: {neural_report["best_neural_model"]["model_name"]}')
    print(f'  F1 Score:   {neural_report["best_neural_model"]["test_f1"]:.4f}')
    print(f'  Recall:     {neural_report["best_neural_model"]["recall"]:.4f}')
    print(f'  Size:       {neural_report["best_neural_model"]["size_mb"]:.2f} MB')
    
    if nn_model_path.exists():
        try:
            # Load with custom objects
            neural_model = keras.models.load_model(
                nn_model_path,
                custom_objects={'F1Score': F1Score}
            )
            print(f'  Model loaded: {nn_model_path.name}')
        except Exception as e:
            print(f'  ⚠️ Error loading model: {e}')
            print(f'  Trying without compilation...')
            try:
                neural_model = keras.models.load_model(nn_model_path, compile=False)
                print(f'  ✓ Model loaded without compilation')
            except Exception as e2:
                print(f'  ❌ Failed to load model: {e2}')
                neural_model = None


# Load transfer learning results from 08 (optional)
transfer_report = RESULTS_DIR / '08_transfer_learning_report.json'
if transfer_report.exists():
    with open(transfer_report, 'r') as f:
        tl_report = json.load(f)
    print(f'\n✓ Transfer Learning Report (Notebook 08):')
    print(f'  YAMNet F1:  {tl_report["yamnet_results"]["test_f1"]:.4f}')
    print(f'  Best model: {tl_report["best_overall_model"]["name"]}')
else:
    tl_report = None
    print(f'\nℹ️ Transfer learning not evaluated yet (Notebook 08)')

print('='*70)


LOADING BEST MODELS...
✓ Classical ML Report (Notebook 06a):
  Best model: XGBoost
  F1 Score:   0.9877
  Recall:     0.9955
  Inference:  0.013 ms
  Size:       1.57 MB

✓ Feature Reduction Results (Notebook 06b):
Configuration  Features  Test F1   Recall  FNR (%)  Est. Extraction (ms)  Model Size (KB)
         Full       260 0.987652 0.995547 0.445254                   330      1611.400391
         FAST        30 0.978856 0.988464 1.153613                    75       465.495117
     BALANCED        50 0.987731 0.993928 0.607165                    95       808.692383
     ACCURATE        80 0.990744 0.996559 0.344060                   140      1267.004883

→ Selected: ACCURATE (80 features) for deployment
   F1: 0.9907
   Recall: 0.9966
   Features: 80
   ✓ Loaded 80 feature indices

  ✓ Model loaded: xgboost_accurate_80_combined.pkl
  ✓ Scaler loaded: scaler_combined.pkl

✓ Neural Network Report (Notebook 07 - 3ch):
  Best model: CNN_LSTM
  F1 Score:   0.9799
  Recall:     0.9844
  

## Section 3: Classical ML Deployment Analysis (emlearn)

### 3.1 Feature Extraction Memory Calculation

In [10]:
print('\nCLASSICAL ML: MEMORY REQUIREMENTS')
print('='*70)

# Feature extraction memory budget
# Based on feature importance from 06b

if classical_report:
    n_features = classical_report['best_model']['n_features_used']
    print(f'Number of features required: {n_features}')
else:
    n_features = 50  # Estimate
    print(f'Estimated features: {n_features}')

# Memory breakdown for classical ML pipeline
classical_memory = {
    # Audio buffers
    'audio_buffer_int16': AUDIO_CONFIG['window_samples'] * 2,  # 16-bit samples
    'audio_buffer_float': AUDIO_CONFIG['window_samples'] * 4,  # float32 for processing
    
    # FFT/STFT intermediate buffers
    'fft_complex': AUDIO_CONFIG['n_fft'] * 8,  # complex64
    'window_buffer': AUDIO_CONFIG['n_fft'] * 4,  # float32
    
    # Mel spectrogram
    'mel_spectrogram': AUDIO_CONFIG['n_frames'] * AUDIO_CONFIG['n_mels'] * 4,  # float32
    
    # Feature vector
    'feature_vector': n_features * 4,  # float32
    
    # Working buffers (means, stds, etc.)
    'working_buffers': 10 * 1024,  # 10 KB estimate
}

# Model memory (depends on model type)
if classical_model:
    if hasattr(classical_model, 'n_estimators'):  # RandomForest/XGBoost
        # Rough estimate: trees * depth * nodes * 4 bytes
        n_trees = getattr(classical_model, 'n_estimators', 100)
        max_depth = getattr(classical_model, 'max_depth', 10)
        nodes_per_tree = 2 ** (max_depth + 1) - 1
        classical_memory['model_size'] = n_trees * nodes_per_tree * 12  # ~12 bytes per node
    else:
        classical_memory['model_size'] = 50 * 1024  # 50 KB estimate
else:
    classical_memory['model_size'] = 50 * 1024

# Inference output
classical_memory['inference_output'] = 8  # probability + label

# Calculate totals
classical_total_kb = sum(classical_memory.values()) / 1024
classical_peak_kb = classical_total_kb * 1.2  # 20% overhead for safety

print(f'\nClassical ML Memory Breakdown (KB):')
print(f'  Audio buffers:     {(classical_memory["audio_buffer_int16"] + classical_memory["audio_buffer_float"])/1024:.1f} KB')
print(f'  FFT/STFT:          {(classical_memory["fft_complex"] + classical_memory["window_buffer"])/1024:.1f} KB')
print(f'  Mel spectrogram:   {classical_memory["mel_spectrogram"]/1024:.1f} KB')
print(f'  Feature vector:    {classical_memory["feature_vector"]/1024:.1f} KB')
print(f'  Working buffers:   {classical_memory["working_buffers"]/1024:.1f} KB')
print(f'  Model size:        {classical_memory["model_size"]/1024:.1f} KB')
print(f'  ───────────────────────────────')
print(f'  Total memory:      {classical_total_kb:.1f} KB')
print(f'  Peak (w/ 20% pad): {classical_peak_kb:.1f} KB')

# Check if fits in SRAM
if classical_peak_kb < ESP32_CONSTRAINTS['available_sram_kb']:
    print(f'\n✅ FITS IN SRAM ({classical_peak_kb:.0f} / {ESP32_CONSTRAINTS["available_sram_kb"]} KB)')
    classical_fits_sram = True
else:
    print(f'\n⚠️ EXCEEDS SRAM ({classical_peak_kb:.0f} / {ESP32_CONSTRAINTS["available_sram_kb"]} KB)')
    print(f'   Overage: {classical_peak_kb - ESP32_CONSTRAINTS["available_sram_kb"]:.0f} KB')
    classical_fits_sram = False

print('='*70)


CLASSICAL ML: MEMORY REQUIREMENTS
Number of features required: 80

Classical ML Memory Breakdown (KB):
  Audio buffers:     93.8 KB
  FFT/STFT:          6.0 KB
  Mel spectrogram:   15.6 KB
  Feature vector:    0.3 KB
  Working buffers:   10.0 KB
  Model size:        896.5 KB
  ───────────────────────────────
  Total memory:      1022.2 KB
  Peak (w/ 20% pad): 1226.6 KB

⚠️ EXCEEDS SRAM (1227 / 432 KB)
   Overage: 795 KB


### 3.2 Classical ML Deployment Framework (emlearn)

In [13]:
print('\nCLASSICAL ML DEPLOYMENT: emlearn')
print('='*70)

print('\nemlearn - Machine Learning for Microcontrollers')
print('  Repository: https://github.com/emlearn/emlearn')
print('  License: MIT')
print('\nKey Features:')
print('  ✓ Converts scikit-learn models to C99 code')
print('  ✓ No dynamic memory allocation')
print('  ✓ Single header file include')
print('  ✓ No external dependencies (no libc required)')
print('  ✓ Tested on ESP32, ESP8266, ARM Cortex-M, AVR')
print('\nSupported Models:')
print('  - RandomForestClassifier / Regressor')
print('  - DecisionTreeClassifier / Regressor')
print('  - ExtraTreesClassifier / Regressor')
print('  - XGBoost (via sklearn API)')
print('  - GaussianNB')
print('  - MLPClassifier / Regressor')
print('\nFeature Extraction Support:')
print('  - Mel-frequency spectrogram (eml_audio)')
print('  - Custom feature extraction in C')

# Try to import emlearn
try:
    import emlearn
    print(f'\n✅ emlearn is installed (version: {emlearn.__version__})')
    emlearn_available = True
    
    # Try converting model
    if classical_model:
        print('\nConverting model to C code...')
        try:
            cmodel = emlearn.convert(classical_model, method='inline')
            
            # Save to temporary file to check size
            temp_c_file = TFLITE_DIR / 'classical_model.h'
            cmodel.save(file=str(temp_c_file), name='grinder_classifier')
            
            c_file_size_kb = temp_c_file.stat().st_size / 1024
            print(f'✅ Model converted successfully')
            print(f'   C header size: {c_file_size_kb:.1f} KB')
            
            # Estimate compiled binary size (typically 30-50% of source)
            estimated_binary_kb = c_file_size_kb * 0.4
            print(f'   Estimated binary: {estimated_binary_kb:.1f} KB')
            
        except Exception as e:
            print(f'⚠️ Conversion failed: {e}')
            # Use model size from report as fallback
            if classical_report:
                estimated_binary_kb = classical_report['best_model'].get('size_kb', 50)
            else:
                estimated_binary_kb = 50
    else:
        print('\n⚠️ No model loaded to convert')
        estimated_binary_kb = 50
        
except ImportError:
    print('\n⚠️ emlearn not installed')
    print('   Install: pip install emlearn')
    emlearn_available = False
    estimated_binary_kb = 50

print('\nDeployment Strategy:')
print('  1. Train model in Python (scikit-learn/XGBoost)')
print('  2. Convert to C with emlearn')
print('  3. Implement feature extraction in C/C++')
print('     - Use emlearn eml_audio for Mel spectrogram')
print('     - Or custom FFT/feature code')
print('  4. Compile & flash to ESP32-S3')
print('  5. Real-time inference in <5ms')

print('='*70)



CLASSICAL ML DEPLOYMENT: emlearn

emlearn - Machine Learning for Microcontrollers
  Repository: https://github.com/emlearn/emlearn
  License: MIT

Key Features:
  ✓ Converts scikit-learn models to C99 code
  ✓ No dynamic memory allocation
  ✓ Single header file include
  ✓ No external dependencies (no libc required)
  ✓ Tested on ESP32, ESP8266, ARM Cortex-M, AVR

Supported Models:
  - RandomForestClassifier / Regressor
  - DecisionTreeClassifier / Regressor
  - ExtraTreesClassifier / Regressor
  - XGBoost (via sklearn API)
  - GaussianNB
  - MLPClassifier / Regressor

Feature Extraction Support:
  - Mel-frequency spectrogram (eml_audio)
  - Custom feature extraction in C

⚠️ emlearn not installed
   Install: pip install emlearn

Deployment Strategy:
  1. Train model in Python (scikit-learn/XGBoost)
  2. Convert to C with emlearn
  3. Implement feature extraction in C/C++
     - Use emlearn eml_audio for Mel spectrogram
     - Or custom FFT/feature code
  4. Compile & flash to ESP32-S

## Section 4: Neural Network Deployment Analysis (TFLite Micro)

### 4.1 Load Test Data for Quantization

In [15]:
if neural_model:
    print('\nLOADING DATA FOR NEURAL NETWORK QUANTIZATION...')
    print('='*70)
    
    # Load spectrograms
    if nn_feature_type == '3ch':
        spec_file = FEATURES_DIR / 'neural' / 'spectrograms_3ch.npy'
    else:
        spec_file = FEATURES_DIR / 'neural' /'spectrograms_1ch.npy'
    
    labels_file = FEATURES_DIR / 'neural' /'labels.npy'
    
    print(f'Loading {spec_file.name}...')
    X_spec = np.load(spec_file)
    y = np.load(labels_file)
    
    print(f'✓ Data loaded: {X_spec.shape}')
    
    # Same split as notebook 07
    X_temp, X_test, y_temp, y_test = train_test_split(
        X_spec, y, test_size=0.15, random_state=SEED, stratify=y
    )
    
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.176, random_state=SEED, stratify=y_temp
    )
    
    # Normalize
    def norm_sample(x):
        return (x - x.mean()) / (x.std() + 1e-8)
    
    X_train_norm = np.array([norm_sample(x) for x in X_train], dtype=np.float32)
    X_test_norm = np.array([norm_sample(x) for x in X_test], dtype=np.float32)
    
    # Reshape if needed
    if len(X_test_norm.shape) == 4:
        N_test, T, F, C = X_test_norm.shape
        N_train = X_train_norm.shape[0]
        X_test_final = X_test_norm.reshape(N_test, T, F * C)
        X_train_final = X_train_norm.reshape(N_train, T, F * C)
    else:
        X_test_final = X_test_norm
        X_train_final = X_train_norm
    
    print(f'Train: {X_train_final.shape[0]:,} samples')
    print(f'Test:  {X_test_final.shape[0]:,} samples')
    
    # Cleanup
    del X_spec, X_temp, X_train, X_val
    del X_train_norm, X_test_norm
    gc.collect()
    
    print('='*70)
else:
    print('⚠️ Neural model not loaded, skipping data preparation')


LOADING DATA FOR NEURAL NETWORK QUANTIZATION...
Loading spectrograms_3ch.npy...
✓ Data loaded: (60348, 101, 40, 3)
Train: 42,267 samples
Test:  9,053 samples


### 4.2 TFLite INT8 Quantization

In [17]:
if neural_model:
    print('\nTFLITE INT8 QUANTIZATION...')
    print('='*70)
    
    # Representative dataset for calibration
    def representative_dataset_gen():
        num_samples = min(500, len(X_train_final))
        for i in range(num_samples):
            yield [X_train_final[i:i+1].astype(np.float32)]
    
    print('Creating INT8 quantized model...')
    print(f'Using {min(500, len(X_train_final))} calibration samples')
    
    # Strategy 1: Try INT8 with experimental flags
    try:
        print('\nAttempt 1: INT8 with experimental flags...')
        converter = tf.lite.TFLiteConverter.from_keras_model(neural_model)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = representative_dataset_gen
        
        # Experimental flags for LSTM compatibility
        converter.experimental_enable_resource_variables = True
        converter._experimental_lower_tensor_list_ops = False
        
        # Target INT8
        converter.target_spec.supported_ops = [
            tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
            tf.lite.OpsSet.SELECT_TF_OPS  # Allow some TF ops if needed
        ]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8
        
        print('Converting...')
        tflite_int8 = converter.convert()
        conversion_method = 'INT8 with TF ops'
        print(f'✅ Success with {conversion_method}')
        
    except Exception as e1:
        print(f'❌ Failed: {e1}')
        
        # Strategy 2: Try INT8 with only built-in ops (no SELECT_TF_OPS)
        try:
            print('\nAttempt 2: INT8 built-ins only...')
            converter = tf.lite.TFLiteConverter.from_keras_model(neural_model)
            converter.optimizations = [tf.lite.Optimize.DEFAULT]
            converter.representative_dataset = representative_dataset_gen
            converter.experimental_enable_resource_variables = True
            
            # Only built-in ops
            converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
            converter.inference_input_type = tf.int8
            converter.inference_output_type = tf.int8
            
            print('Converting...')
            tflite_int8 = converter.convert()
            conversion_method = 'INT8 built-ins'
            print(f'✅ Success with {conversion_method}')
            
        except Exception as e2:
            print(f'❌ Failed: {e2}')
            
            # Strategy 3: Dynamic range quantization (weights only)
            try:
                print('\nAttempt 3: Dynamic range quantization (fallback)...')
                converter = tf.lite.TFLiteConverter.from_keras_model(neural_model)
                converter.optimizations = [tf.lite.Optimize.DEFAULT]
                # No representative dataset - only weight quantization
                
                print('Converting...')
                tflite_int8 = converter.convert()
                conversion_method = 'Dynamic range (weights INT8, activations FP32)'
                print(f'✅ Success with {conversion_method}')
                print('⚠️  Note: Activations still FP32 (not ideal for embedded)')
                
            except Exception as e3:
                print(f'❌ All quantization attempts failed')
                print(f'   Error 1 (INT8+TF): {str(e1)[:100]}')
                print(f'   Error 2 (INT8 only): {str(e2)[:100]}')
                print(f'   Error 3 (Dynamic): {str(e3)[:100]}')
                print('\n⚠️  Model architecture may not be compatible with TFLite')
                print('   Consider: Rebuilding model without LSTM, or using simpler architecture')
                tflite_int8 = None
    
    if tflite_int8:
        # Save
        tflite_int8_path = TFLITE_DIR / f'model_{nn_feature_type}_int8.tflite'
        with open(tflite_int8_path, 'wb') as f:
            f.write(tflite_int8)
        
        int8_size_kb = len(tflite_int8) / 1024
        
        print(f'\n✅ INT8 model saved: {tflite_int8_path.name}')
        print(f'  Size: {int8_size_kb:.1f} KB')
        print(f'  Method: {conversion_method}')
        
        # Evaluate INT8 model
        print('\nEvaluating INT8 model...')
        interpreter_int8 = tf.lite.Interpreter(model_path=str(tflite_int8_path))
        interpreter_int8.allocate_tensors()
        
        input_details_int8 = interpreter_int8.get_input_details()
        output_details_int8 = interpreter_int8.get_output_details()
        
        # Check if inputs/outputs are quantized
        input_is_quantized = input_details_int8[0]['dtype'] in [np.int8, np.uint8]
        output_is_quantized = output_details_int8[0]['dtype'] in [np.int8, np.uint8]
        
        print(f'  Input type:  {input_details_int8[0]["dtype"]} {"(quantized)" if input_is_quantized else "(float)"}')
        print(f'  Output type: {output_details_int8[0]["dtype"]} {"(quantized)" if output_is_quantized else "(float)"}')
        
        if input_is_quantized:
            input_scale, input_zero_point = input_details_int8[0]['quantization']
        else:
            input_scale, input_zero_point = 1.0, 0
            
        if output_is_quantized:
            output_scale, output_zero_point = output_details_int8[0]['quantization']
        else:
            output_scale, output_zero_point = 1.0, 0
        
        # Run inference
        y_pred_int8 = []
        inference_times_int8 = []
        
        for i in range(len(X_test_final)):
            input_data = X_test_final[i:i+1].astype(np.float32)
            
            # Quantize if needed
            if input_is_quantized:
                input_data = (input_data / input_scale + input_zero_point).astype(np.int8)
            
            interpreter_int8.set_tensor(input_details_int8[0]['index'], input_data)
            
            start = time.time()
            interpreter_int8.invoke()
            inference_times_int8.append((time.time() - start) * 1000)
            
            output_data = interpreter_int8.get_tensor(output_details_int8[0]['index'])
            
            # Dequantize if needed
            if output_is_quantized:
                output_data = (output_data.astype(np.float32) - output_zero_point) * output_scale
            
            y_pred_int8.append(output_data[0][0])
        
        y_pred_int8 = np.array(y_pred_int8)
        y_pred_int8_binary = (y_pred_int8 > 0.5).astype(int)
        
        # Metrics
        int8_acc = accuracy_score(y_test, y_pred_int8_binary)
        int8_f1 = f1_score(y_test, y_pred_int8_binary)
        int8_recall = recall_score(y_test, y_pred_int8_binary)
        int8_precision = precision_score(y_test, y_pred_int8_binary)
        int8_inference_ms = np.mean(inference_times_int8)
        
        print(f'\nINT8 Performance:')
        print(f'  F1:        {int8_f1:.4f}')
        print(f'  Recall:    {int8_recall:.4f}')
        print(f'  Precision: {int8_precision:.4f}')
        print(f'  Inference: {int8_inference_ms:.2f} ms (CPU)')
        print(f'  Est. ESP32: {int8_inference_ms * 2:.2f} ms (conservative)')
        
        # Original performance
        original_f1 = neural_report['best_neural_model']['test_f1']
        f1_loss = original_f1 - int8_f1
        
        print(f'\nAccuracy Loss:')
        print(f'  F1 loss: {f1_loss:.4f} ({f1_loss/original_f1*100:.1f}%)')
        
        if f1_loss < 0.02:
            print(f'  ✅ Acceptable (<2%)')
            int8_acceptable = True
        else:
            print(f'  ⚠️ Significant (>2%) - consider INT16x8 or model redesign')
            int8_acceptable = False
        
    else:
        # Quantization failed completely
        int8_size_kb = 0
        int8_f1 = 0
        int8_recall = 0
        int8_precision = 0
        int8_inference_ms = 0
        int8_acceptable = False
        
        print('\n❌ Quantization not successful')
        print('   Neural network deployment may not be viable')
        print('   Recommendation: Use Classical ML approach instead')
    
    print('='*70)
else:
    int8_size_kb = 0
    int8_f1 = 0
    int8_acceptable = False



TFLITE INT8 QUANTIZATION...
Creating INT8 quantized model...
Using 500 calibration samples

Attempt 1: INT8 with experimental flags...
Converting...
INFO:tensorflow:Assets written to: /var/folders/41/zp1kfbg927lf6brpdz24gbd00000gn/T/tmp96n5dpy0/assets


INFO:tensorflow:Assets written to: /var/folders/41/zp1kfbg927lf6brpdz24gbd00000gn/T/tmp96n5dpy0/assets


Saved artifact at '/var/folders/41/zp1kfbg927lf6brpdz24gbd00000gn/T/tmp96n5dpy0'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 101, 120), dtype=tf.float32, name='input_layer_3')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  5692735328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5692732864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5692793776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5692794832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5692780032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5692793600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5692925904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5692941056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5693010864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5693011920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5692988928: TensorSpec

W0000 00:00:1765390245.116626  233603 tf_tfl_flatbuffer_helpers.cc:392] Ignored output_format.
W0000 00:00:1765390245.116635  233603 tf_tfl_flatbuffer_helpers.cc:395] Ignored drop_control_dependency.
2025-12-10 18:10:45.116717: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/41/zp1kfbg927lf6brpdz24gbd00000gn/T/tmp96n5dpy0
2025-12-10 18:10:45.117341: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2025-12-10 18:10:45.117345: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /var/folders/41/zp1kfbg927lf6brpdz24gbd00000gn/T/tmp96n5dpy0
2025-12-10 18:10:45.123080: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2025-12-10 18:10:45.150890: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: /var/folders/41/zp1kfbg927lf6brpdz24gbd00000gn/T/tmp96n5dpy0
2025-12-10 18:10:45.160671: I tensorflow/cc/saved_model/loader.cc:

✅ Success with INT8 with TF ops

✅ INT8 model saved: model_3ch_int8.tflite
  Size: 144.4 KB
  Method: INT8 with TF ops

Evaluating INT8 model...
  Input type:  <class 'numpy.int8'> (quantized)
  Output type: <class 'numpy.int8'> (quantized)

INT8 Performance:
  F1:        0.9731
  Recall:    0.9654
  Precision: 0.9809
  Inference: 0.21 ms (CPU)
  Est. ESP32: 0.42 ms (conservative)

Accuracy Loss:
  F1 loss: 0.0068 (0.7%)
  ✅ Acceptable (<2%)


### 4.3 TFLite INT16x8 Quantization (if INT8 inadequate)

In [19]:
if neural_model and not int8_acceptable:
    print('\nTFLITE INT16x8 QUANTIZATION...')
    print('='*70)
    print('INT8 accuracy loss too high, trying INT16 activations...')
    
    # Create converter
    converter = tf.lite.TFLiteConverter.from_keras_model(neural_model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset_gen
    
    # INT16 activations, INT8 weights
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.EXPERIMENTAL_TFLITE_BUILTINS_ACTIVATIONS_INT16_WEIGHTS_INT8
    ]
    
    # Convert
    print('Converting...')
    tflite_int16x8 = converter.convert()
    
    # Save
    tflite_int16x8_path = TFLITE_DIR / f'model_{nn_feature_type}_int16x8.tflite'
    with open(tflite_int16x8_path, 'wb') as f:
        f.write(tflite_int16x8)
    
    int16x8_size_kb = len(tflite_int16x8) / 1024
    
    print(f'\n✓ INT16x8 model saved: {tflite_int16x8_path.name}')
    print(f'  Size: {int16x8_size_kb:.1f} KB')
    
    # Evaluate
    interpreter_int16x8 = tf.lite.Interpreter(model_path=str(tflite_int16x8_path))
    interpreter_int16x8.allocate_tensors()
    
    input_details_16 = interpreter_int16x8.get_input_details()
    output_details_16 = interpreter_int16x8.get_output_details()
    
    # Run inference (implementation depends on input/output types)
    y_pred_int16x8 = []
    inference_times_16 = []
    
    for i in range(len(X_test_final)):
        input_data = X_test_final[i:i+1].astype(np.float32)
        
        # Handle quantization based on actual input type
        if input_details_16[0]['dtype'] == np.int16:
            scale, zp = input_details_16[0]['quantization']
            input_q = (input_data / scale + zp).astype(np.int16)
        else:
            input_q = input_data
        
        interpreter_int16x8.set_tensor(input_details_16[0]['index'], input_q)
        
        start = time.time()
        interpreter_int16x8.invoke()
        inference_times_16.append((time.time() - start) * 1000)
        
        output_q = interpreter_int16x8.get_tensor(output_details_16[0]['index'])
        
        # Dequantize if needed
        if output_details_16[0]['dtype'] in [np.int16, np.int8]:
            scale, zp = output_details_16[0]['quantization']
            output_data = (output_q.astype(np.float32) - zp) * scale
        else:
            output_data = output_q
        
        y_pred_int16x8.append(output_data[0][0])
    
    y_pred_int16x8 = np.array(y_pred_int16x8)
    y_pred_int16x8_binary = (y_pred_int16x8 > 0.5).astype(int)
    
    # Metrics
    int16x8_f1 = f1_score(y_test, y_pred_int16x8_binary)
    int16x8_recall = recall_score(y_test, y_pred_int16x8_binary)
    int16x8_inference_ms = np.mean(inference_times_16)
    
    print(f'\nINT16x8 Performance:')
    print(f'  F1:        {int16x8_f1:.4f}')
    print(f'  Recall:    {int16x8_recall:.4f}')
    print(f'  Inference: {int16x8_inference_ms:.2f} ms (CPU)')
    print(f'  Est. ESP32: {int16x8_inference_ms * 2:.2f} ms')
    
    f1_loss_16 = original_f1 - int16x8_f1
    print(f'\nF1 loss: {f1_loss_16:.4f} ({f1_loss_16/original_f1*100:.1f}%)')
    
    if int16x8_f1 > int8_f1:
        print('✅ INT16x8 outperforms INT8')
    
    print('='*70)
elif neural_model:
    print('\n✓ INT8 quantization sufficient, skipping INT16x8')
    int16x8_size_kb = 0
    int16x8_f1 = 0
else:
    int16x8_size_kb = 0
    int16x8_f1 = 0


✓ INT8 quantization sufficient, skipping INT16x8


### 4.4 Neural Network Memory Requirements

In [18]:
if neural_model:
    print('\nNEURAL NETWORK MEMORY REQUIREMENTS')
    print('='*70)
    
    # Memory breakdown
    nn_memory = {
        # Audio buffer (raw)
        'audio_buffer': AUDIO_CONFIG['window_samples'] * 2,  # INT16
        
        # Spectrogram preprocessing
        'fft_buffer': AUDIO_CONFIG['n_fft'] * 8,  # complex64
        'mel_filters': AUDIO_CONFIG['n_mels'] * AUDIO_CONFIG['n_fft'] // 2 * 4,  # float32
        
        # Spectrogram output
        'spectrogram_1ch': AUDIO_CONFIG['n_frames'] * AUDIO_CONFIG['n_mels'] * 4,
    }
    
    if nn_feature_type == '3ch':
        nn_memory['spectrogram_3ch'] = nn_memory['spectrogram_1ch'] * 3
        del nn_memory['spectrogram_1ch']
    
    # TFLite Runtime overhead
    nn_memory['tflite_runtime'] = 40 * 1024  # ~40 KB
    
    # Tensor arena (CRITICAL!)
    # Estimate based on model architecture
    # Rule of thumb: 2-3x largest layer output + model weights in memory
    model_params = neural_model.count_params()
    
    # Conservative estimate
    if int8_size_kb > 0:
        # INT8: 1 byte per param + activations
        estimated_arena_kb = (model_params / 1024) + 150  # +150 KB for activations
    else:
        estimated_arena_kb = 350
    
    nn_memory['tensor_arena'] = estimated_arena_kb * 1024
    
    # Model in flash (not SRAM, but track it)
    nn_memory['model_flash'] = int8_size_kb * 1024
    
    # Calculate SRAM usage (without model, as it's in flash)
    nn_sram_kb = (sum(nn_memory.values()) - nn_memory['model_flash']) / 1024
    nn_peak_kb = nn_sram_kb * 1.15  # 15% safety margin
    
    print(f'\nNeural Network Memory Breakdown (KB):')
    print(f'  Audio buffer:      {nn_memory["audio_buffer"]/1024:.1f} KB')
    print(f'  FFT/preprocessing: {(nn_memory["fft_buffer"] + nn_memory["mel_filters"])/1024:.1f} KB')
    if '3ch' in nn_feature_type:
        print(f'  Spectrogram (3ch): {nn_memory["spectrogram_3ch"]/1024:.1f} KB')
    else:
        print(f'  Spectrogram (1ch): {nn_memory["spectrogram_1ch"]/1024:.1f} KB')
    print(f'  TFLite runtime:    {nn_memory["tflite_runtime"]/1024:.1f} KB')
    print(f'  Tensor arena:      {nn_memory["tensor_arena"]/1024:.1f} KB  ⚠️ CRITICAL')
    print(f'  ───────────────────────────────')
    print(f'  Total SRAM:        {nn_sram_kb:.1f} KB')
    print(f'  Peak (w/ 15% pad): {nn_peak_kb:.1f} KB')
    print(f'  ───────────────────────────────')
    print(f'  Model (Flash):     {nn_memory["model_flash"]/1024:.1f} KB')
    
    # Check constraints
    if nn_peak_kb < ESP32_CONSTRAINTS['available_sram_kb']:
        print(f'\n✅ FITS IN SRAM ({nn_peak_kb:.0f} / {ESP32_CONSTRAINTS["available_sram_kb"]} KB)')
        nn_fits_sram = True
    else:
        print(f'\n❌ EXCEEDS SRAM ({nn_peak_kb:.0f} / {ESP32_CONSTRAINTS["available_sram_kb"]} KB)')
        print(f'   Overage: {nn_peak_kb - ESP32_CONSTRAINTS["available_sram_kb"]:.0f} KB')
        print(f'\n   Options:')
        print(f'   1. Use PSRAM (slower, ~40-60× inference time)')
        print(f'   2. Reduce model complexity')
        print(f'   3. Use 1ch instead of 3ch spectrograms')
        nn_fits_sram = False
    
    if nn_memory['tensor_arena'] / 1024 > ESP32_CONSTRAINTS['target_tensor_arena_kb']:
        print(f'\n⚠️ Tensor arena exceeds target:')
        print(f'   {nn_memory["tensor_arena"]/1024:.0f} / {ESP32_CONSTRAINTS["target_tensor_arena_kb"]} KB')
    
    print('='*70)
else:
    nn_peak_kb = 0
    nn_fits_sram = False


NEURAL NETWORK MEMORY REQUIREMENTS

Neural Network Memory Breakdown (KB):
  Audio buffer:      31.2 KB
  FFT/preprocessing: 44.0 KB
  Spectrogram (3ch): 46.9 KB
  TFLite runtime:    40.0 KB
  Tensor arena:      263.0 KB  ⚠️ CRITICAL
  ───────────────────────────────
  Total SRAM:        425.1 KB
  Peak (w/ 15% pad): 488.9 KB
  ───────────────────────────────
  Model (Flash):     144.4 KB

❌ EXCEEDS SRAM (489 / 432 KB)
   Overage: 57 KB

   Options:
   1. Use PSRAM (slower, ~40-60× inference time)
   2. Reduce model complexity
   3. Use 1ch instead of 3ch spectrograms


## Section 5: Comprehensive Comparison & Recommendation

In [20]:
print('\nDEPLOYMENT COMPARISON')
print('='*70)

# Create comparison table
comparison_data = []

# Classical ML
if classical_report:
    comparison_data.append({
        'Approach': 'Classical ML (emlearn)',
        'Model': classical_report['best_model']['name'],
        'F1 Score': classical_report['best_model']['test_f1'],
        'Recall': classical_report['best_model']['recall'],
        'Model Size (KB)': estimated_binary_kb,
        'SRAM Usage (KB)': classical_peak_kb,
        'Inference (ms)': '<5',
        'Fits SRAM': '✅' if classical_fits_sram else '❌',
        'Deployment': 'emlearn C99',
    })

# Neural Network INT8
if neural_model and int8_size_kb > 0:
    comparison_data.append({
        'Approach': f'Neural Network (TFLite INT8)',
        'Model': f'{neural_report["best_neural_model"]["model_name"]} ({nn_feature_type})',
        'F1 Score': int8_f1,
        'Recall': int8_recall,
        'Model Size (KB)': int8_size_kb,
        'SRAM Usage (KB)': nn_peak_kb,
        'Inference (ms)': f'{int8_inference_ms * 2:.1f}',
        'Fits SRAM': '✅' if nn_fits_sram else '❌',
        'Deployment': 'TFLite Micro',
    })

# Neural Network INT16x8
if neural_model and int16x8_size_kb > 0:
    comparison_data.append({
        'Approach': f'Neural Network (TFLite INT16x8)',
        'Model': f'{neural_report["best_neural_model"]["model_name"]} ({nn_feature_type})',
        'F1 Score': int16x8_f1,
        'Recall': int16x8_recall,
        'Model Size (KB)': int16x8_size_kb,
        'SRAM Usage (KB)': nn_peak_kb * 1.1,  # Slightly more for INT16
        'Inference (ms)': f'{int16x8_inference_ms * 2:.1f}',
        'Fits SRAM': '❌' if nn_peak_kb * 1.1 > ESP32_CONSTRAINTS['available_sram_kb'] else '✅',
        'Deployment': 'TFLite Micro',
    })

df_comparison = pd.DataFrame(comparison_data)

print('\n')
print(df_comparison.to_string(index=False))
print('\n')

# Save comparison
comparison_csv = RESULTS_DIR / '09_deployment_comparison.csv'
df_comparison.to_csv(comparison_csv, index=False)
print(f'✓ Saved: {comparison_csv.name}')

print('='*70)


DEPLOYMENT COMPARISON


                    Approach          Model  F1 Score   Recall  Model Size (KB)  SRAM Usage (KB) Inference (ms) Fits SRAM   Deployment
      Classical ML (emlearn)        XGBoost  0.990744 0.996559        50.000000      1226.615625             <5         ❌  emlearn C99
Neural Network (TFLite INT8) CNN_LSTM (3ch)  0.973072 0.965392       144.445312       488.894873            0.4         ❌ TFLite Micro


✓ Saved: 09_deployment_comparison.csv


### 5.1 Automated Recommendation

In [ ]:
print('\nAUTOMATED DEPLOYMENT RECOMMENDATION')
print('='*70)

# Scoring system
scores = {}

for idx, row in df_comparison.iterrows():
    approach = row['Approach']
    score = 0
    reasons = []
    
    # Accuracy (30 points)
    f1 = row['F1 Score']
    if f1 >= 0.95:
        score += 30
        reasons.append('Excellent F1 (≥0.95)')
    elif f1 >= 0.90:
        score += 25
        reasons.append('Good F1 (≥0.90)')
    else:
        score += 15
        reasons.append('Acceptable F1')
    
    # Recall (30 points) - CRITICAL for security
    recall = row['Recall']
    if recall >= 0.95:
        score += 30
        reasons.append('Excellent Recall (≥0.95)')
    elif recall >= 0.90:
        score += 25
        reasons.append('Good Recall (≥0.90)')
    else:
        score += 15
        reasons.append('Acceptable Recall')
    
    # Memory fit (25 points)
    if row['Fits SRAM'] == '✅':
        score += 25
        reasons.append('Fits in SRAM')
    else:
        score += 0
        reasons.append('❌ Exceeds SRAM')
    
    # Inference speed (15 points)
    inference_str = str(row['Inference (ms)']).replace('<', '')
    inference_ms = float(inference_str)
    
    if inference_ms < 10:
        score += 15
        reasons.append('Very fast inference (<10ms)')
    elif inference_ms < 50:
        score += 10
        reasons.append('Fast inference (<50ms)')
    else:
        score += 5
        reasons.append('Slow inference (≥50ms)')
    
    scores[approach] = {'score': score, 'reasons': reasons}

# Rank approaches
ranked = sorted(scores.items(), key=lambda x: x[1]['score'], reverse=True)

print('\nRanking (out of 100 points):\n')
for i, (approach, data) in enumerate(ranked, 1):
    print(f'{i}. {approach}: {data["score"]} points')
    for reason in data['reasons']:
        print(f'   • {reason}')
    print()

# Best approach
best_approach = ranked[0][0]
best_score = ranked[0][1]['score']

print('='*70)
print(f'🏆 RECOMMENDED APPROACH: {best_approach}')
print(f'   Score: {best_score}/100')
print('='*70)

# Detailed recommendation
if 'Classical' in best_approach:
    print('\n✅ Deploy with Classical ML (emlearn):')
    print('   1. Use emlearn to convert XGBoost/RandomForest to C')
    print('   2. Implement feature extraction in C/C++')
    print('   3. Very low memory footprint (<150 KB)')
    print('   4. Ultra-fast inference (<5 ms)')
    print('   5. No external runtime dependencies')
    print('\n   Pros:')
    print('   • Extremely efficient')
    print('   • Easy to debug (interpretable trees)')
    print('   • Proven on ESP32')
    print('\n   Cons:')
    print('   • Feature extraction must be ported to C')
    print('   • May have slightly lower accuracy than NN')
elif 'INT8' in best_approach:
    print('\n✅ Deploy with Neural Network (TFLite INT8):')
    print('   1. Use quantized model from this notebook')
    print('   2. Implement spectrogram preprocessing')
    print('   3. TFLite Micro runtime')
    print('   4. Good balance of accuracy vs size')
    print('\n   Pros:')
    print('   • High accuracy')
    print('   • Standard deployment path (TFLite)')
    print('   • Good quantization (minimal accuracy loss)')
    print('\n   Cons:')
    print('   • Higher memory usage (tensor arena)')
    print('   • Slower inference (~20-40ms)')
    print('   • More complex debugging')
elif 'INT16x8' in best_approach:
    print('\n⚠️ Deploy with Neural Network (TFLite INT16x8):')
    print('   1. Better accuracy than INT8')
    print('   2. Larger memory footprint')
    print('   3. May need careful memory management')
    print('\n   Consider:')
    print('   • INT8 had too much accuracy loss')
    print('   • Ensure tensor arena fits in SRAM')
    print('   • Monitor peak memory usage')

print('\n' + '='*70)

## Section 6: Generate Deployment Package

In [ ]:
print('\nGENERATING DEPLOYMENT PACKAGE...')
print('='*70)

# Create deployment directory
deploy_dir = PROJECT_ROOT / 'deployment' / 'esp32_s3_final'
deploy_dir.mkdir(parents=True, exist_ok=True)

# Copy best model
if 'Classical' in best_approach:
    # Classical ML
    if emlearn_available and classical_model:
        # Save emlearn C header
        cmodel = emlearn.convert(classical_model, method='inline')
        c_header = deploy_dir / 'grinder_classifier.h'
        cmodel.save(file=str(c_header), name='grinder_classifier')
        print(f'✓ Saved emlearn C header: {c_header.name}')
    
    deployment_type = 'classical_ml'
    deployment_framework = 'emlearn'
    
else:
    # Neural Network
    import shutil
    
    if 'INT16x8' in best_approach and int16x8_size_kb > 0:
        src = tflite_int16x8_path
        quant_type = 'int16x8'
    else:
        src = tflite_int8_path
        quant_type = 'int8'
    
    dst = deploy_dir / 'model.tflite'
    shutil.copy(src, dst)
    print(f'✓ Copied {quant_type} model: {dst.name}')
    
    deployment_type = 'neural_network'
    deployment_framework = 'tflite_micro'

# Create deployment info JSON
deployment_info = {
    'timestamp': datetime.now().isoformat(),
    'recommended_approach': best_approach,
    'deployment_type': deployment_type,
    'framework': deployment_framework,
    'target_device': 'XIAO ESP32-S3 Sense',
    'comparison_results': df_comparison.to_dict('records'),
    'esp32_constraints': ESP32_CONSTRAINTS,
    'audio_config': AUDIO_CONFIG,
}

if 'Classical' in best_approach:
    deployment_info['model_info'] = {
        'type': 'classical_ml',
        'algorithm': classical_report['best_model']['name'],
        'n_features': classical_report['best_model']['n_features_used'],
        'f1_score': classical_report['best_model']['test_f1'],
        'recall': classical_report['best_model']['recall'],
        'estimated_size_kb': estimated_binary_kb,
        'estimated_sram_kb': classical_peak_kb,
        'estimated_inference_ms': 5,
    }
else:
    deployment_info['model_info'] = {
        'type': 'neural_network',
        'architecture': neural_report['best_neural_model']['model_name'],
        'feature_type': nn_feature_type,
        'quantization': quant_type,
        'f1_score': int8_f1 if quant_type == 'int8' else int16x8_f1,
        'recall': int8_recall if quant_type == 'int8' else int16x8_recall,
        'model_size_kb': int8_size_kb if quant_type == 'int8' else int16x8_size_kb,
        'estimated_sram_kb': nn_peak_kb,
        'estimated_inference_ms': int8_inference_ms * 2 if quant_type == 'int8' else int16x8_inference_ms * 2,
    }

# Save deployment info
info_path = deploy_dir / 'deployment_info.json'
with open(info_path, 'w') as f:
    json.dump(deployment_info, f, indent=2)

print(f'✓ Saved deployment info: {info_path.name}')

# Create README
readme_content = f'''# ESP32-S3 Deployment Package

## Recommended Approach: {best_approach}

### Model Performance
- **F1 Score:** {deployment_info["model_info"]["f1_score"]:.4f}
- **Recall:** {deployment_info["model_info"]["recall"]:.4f}
- **False Negative Rate:** {(1 - deployment_info["model_info"]["recall"]) * 100:.2f}%

### Resource Usage
- **Model Size:** {deployment_info["model_info"]["estimated_size_kb" if "estimated_size_kb" in deployment_info["model_info"] else "model_size_kb"]:.0f} KB (Flash)
- **SRAM Usage:** {deployment_info["model_info"]["estimated_sram_kb"]:.0f} KB
- **Inference Time:** ~{deployment_info["model_info"]["estimated_inference_ms"]:.0f} ms

### Deployment Framework
- **Framework:** {deployment_framework}
- **Device:** XIAO ESP32-S3 Sense

### Files
- `deployment_info.json` - Complete deployment configuration
- `{"grinder_classifier.h" if deployment_type == "classical_ml" else "model.tflite"}` - Model file
- `README.md` - This file

### Next Steps
1. Review deployment_info.json for complete specifications
2. Implement feature extraction on ESP32-S3
3. Integrate model with firmware
4. Test on real hardware
5. Optimize power consumption
6. Field testing

Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
'''

readme_path = deploy_dir / 'README.md'
with open(readme_path, 'w') as f:
    f.write(readme_content)

print(f'✓ Created README: {readme_path.name}')

print(f'\nDeployment package: {deploy_dir}')
print('\nContents:')
for item in deploy_dir.iterdir():
    size = item.stat().st_size / 1024 if item.is_file() else 0
    print(f'  • {item.name:<30} {size:>8.1f} KB' if size else f'  • {item.name}')

print('='*70)

## Section 7: Visualization

In [ ]:
print('\nGenerating visualizations...')

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Memory comparison
approaches = [row['Approach'].split('(')[0].strip()[:15] for _, row in df_comparison.iterrows()]
memory_usage = [row['SRAM Usage (KB)'] for _, row in df_comparison.iterrows()]

colors = ['green' if m < ESP32_CONSTRAINTS['available_sram_kb'] else 'red' for m in memory_usage]

axes[0, 0].barh(approaches, memory_usage, color=colors, alpha=0.7)
axes[0, 0].axvline(x=ESP32_CONSTRAINTS['available_sram_kb'], color='red', linestyle='--', 
                   label=f'SRAM Limit ({ESP32_CONSTRAINTS["available_sram_kb"]}KB)', linewidth=2)
axes[0, 0].set_xlabel('SRAM Usage (KB)')
axes[0, 0].set_title('Memory Usage Comparison', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(axis='x', alpha=0.3)

# 2. Performance comparison
f1_scores = [row['F1 Score'] for _, row in df_comparison.iterrows()]
recalls = [row['Recall'] for _, row in df_comparison.iterrows()]

x = np.arange(len(approaches))
width = 0.35

axes[0, 1].bar(x - width/2, f1_scores, width, label='F1 Score', alpha=0.8)
axes[0, 1].bar(x + width/2, recalls, width, label='Recall', alpha=0.8)
axes[0, 1].set_ylabel('Score')
axes[0, 1].set_title('Model Performance', fontweight='bold')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(approaches, rotation=45, ha='right')
axes[0, 1].legend()
axes[0, 1].grid(axis='y', alpha=0.3)
axes[0, 1].set_ylim([0.85, 1.0])

# 3. Model size comparison
model_sizes = []
for _, row in df_comparison.iterrows():
    size = row.get('Model Size (KB)', 0)
    model_sizes.append(size)

axes[1, 0].bar(approaches, model_sizes, alpha=0.7, color='steelblue')
axes[1, 0].axhline(y=ESP32_CONSTRAINTS['target_model_size_kb'], color='red', 
                   linestyle='--', label=f'Target ({ESP32_CONSTRAINTS["target_model_size_kb"]}KB)')
axes[1, 0].set_ylabel('Model Size (KB)')
axes[1, 0].set_title('Model Size Comparison', fontweight='bold')
axes[1, 0].set_xticklabels(approaches, rotation=45, ha='right')
axes[1, 0].legend()
axes[1, 0].grid(axis='y', alpha=0.3)

# 4. Scoring visualization
approach_names = [name.split('(')[0].strip()[:15] for name in scores.keys()]
score_values = [data['score'] for data in scores.values()]

colors_score = ['gold' if i == 0 else 'silver' if i == 1 else 'lightblue' 
                for i in range(len(score_values))]

axes[1, 1].barh(approach_names, score_values, color=colors_score, alpha=0.8)
axes[1, 1].set_xlabel('Score (out of 100)')
axes[1, 1].set_title('Overall Deployment Score', fontweight='bold')
axes[1, 1].grid(axis='x', alpha=0.3)

# Add value labels
for i, v in enumerate(score_values):
    axes[1, 1].text(v + 1, i, str(v), va='center')

plt.tight_layout()
fig_path = FIGURES_DIR / '09_deployment_comparison.png'
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
print(f'✓ Saved: {fig_path.name}')
plt.close()

print('✓ Visualization complete')

## Section 8: Final Report

In [ ]:
print('\nFINAL DEPLOYMENT REPORT')
print('='*70)

report = {
    'timestamp': datetime.now().isoformat(),
    'notebook': '09_deployment_comparison_optimization',
    'recommended_approach': best_approach,
    'recommendation_score': best_score,
    'comparison': df_comparison.to_dict('records'),
    'deployment_package': str(deploy_dir),
    'esp32_constraints': ESP32_CONSTRAINTS,
    'conclusions': [],
}

# Add conclusions
if 'Classical' in best_approach:
    report['conclusions'] = [
        'Classical ML (emlearn) recommended for deployment',
        'Excellent memory efficiency (<150 KB SRAM)',
        'Ultra-fast inference (<5 ms)',
        'Feature extraction must be implemented in C/C++',
        'Proven deployment path for ESP32',
    ]
else:
    report['conclusions'] = [
        'Neural Network (TFLite) recommended for deployment',
        'Higher accuracy than classical ML',
        f'Quantization: {quant_type}',
        'Moderate memory usage (fits in SRAM)' if nn_fits_sram else 'High memory usage (needs optimization)',
        'Spectrogram preprocessing required',
    ]

# Save report
report_path = RESULTS_DIR / '09_deployment_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f'✓ Saved: {report_path.name}')

print('\n' + '='*70)
print('✅ NOTEBOOK 09 COMPLETE')
print('='*70)

print(f'\n🏆 Recommended Deployment: {best_approach}')
print(f'   Score: {best_score}/100')
print(f'\n📦 Deployment package ready: {deploy_dir}')
print(f'\n📊 Full comparison: {comparison_csv.name}')
print(f'📈 Visualization: {fig_path.name}')

print('\n' + '='*70)
print('Next: Notebook 10 - ESP32-S3 Firmware Integration & Testing')
print('='*70)

# Memory cleanup
if 'X_train_final' in locals():
    del X_train_final, X_test_final
gc.collect()
keras.backend.clear_session()
print('\n✓ Memory cleaned up')